# Задание 2

Скалярный autograd на torch без torch.autograd и torch.nn.
backward обходит граф в обратном топологическом порядке и пересчитывает градиенты. Производная ReLU в нуле принимается равной нулю.


In [1]:
import torch
import unittest

In [2]:
class Node:
    def __init__(self, data, _children=(), _op=""):
        self.data = torch.as_tensor(data, dtype=torch.float64)
        self.grad = torch.zeros_like(self.data)
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Node(data={self.data.item():g}, grad={self.grad.item():g})"

    def __add__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        out = Node(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    __radd__ = __add__

    def __mul__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        out = Node(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    __rmul__ = __mul__

    def relu(self):
        out = Node(self.data.clamp(min=0), (self,), "ReLU")

        def _backward():
            self.grad += (self.data > 0) * out.grad

        out._backward = _backward
        return out

    def backward(self):
        order = []
        visited = set()

        def visit(node):
            if node not in visited:
                visited.add(node)
                for child in node._prev:
                    visit(child)
                order.append(node)

        visit(self)
        for node in order:
            node.grad = torch.zeros_like(node.data)
        self.grad = torch.ones_like(self.data)
        for node in reversed(order):
            node._backward()

In [3]:
a = Node(2)
b = Node(-3)
c = Node(10)
d = a + b * c
e = d.relu()
e.backward()
print(a, b, c, d, e)

Node(data=2, grad=0) Node(data=-3, grad=0) Node(data=10, grad=0) Node(data=-28, grad=0) Node(data=0, grad=1)


В примере условия указан неверный результат: при d = -28 ReLU возвращает 0, поэтому градиенты a, b, c и d равны 0, а градиент e равен 1.

In [4]:
class TestNode(unittest.TestCase):
    def test_add(self):
        a, b = Node(2), Node(-3)
        out = a + b
        out.backward()
        self.assertEqual(out.data.item(), -1)
        self.assertEqual(a.grad.item(), 1)
        self.assertEqual(b.grad.item(), 1)

    def test_mul(self):
        a, b = Node(2), Node(-3)
        out = a * b
        out.backward()
        self.assertEqual(out.data.item(), -6)
        self.assertEqual(a.grad.item(), -3)
        self.assertEqual(b.grad.item(), 2)

    def test_relu(self):
        for value, output, gradient in [(2, 2, 1), (-2, 0, 0), (0, 0, 0)]:
            with self.subTest(value=value):
                a = Node(value)
                out = a.relu()
                out.backward()
                self.assertEqual(out.data.item(), output)
                self.assertEqual(a.grad.item(), gradient)

    def test_constants(self):
        a = Node(2)
        out = 2 * a + a * 3 + 4 + a + 5
        out.backward()
        self.assertEqual(out.data.item(), 21)
        self.assertEqual(a.grad.item(), 6)

    def test_same_operand_add(self):
        a = Node(3)
        out = a + a
        out.backward()
        self.assertEqual(out.data.item(), 6)
        self.assertEqual(a.grad.item(), 2)

    def test_same_operand_mul(self):
        a = Node(3)
        out = a * a
        out.backward()
        self.assertEqual(out.data.item(), 9)
        self.assertEqual(a.grad.item(), 6)

    def test_shared_graph(self):
        a = Node(2)
        b = a * a
        out = b + b * a
        out.backward()
        self.assertEqual(out.data.item(), 12)
        self.assertEqual(b.grad.item(), 3)
        self.assertEqual(a.grad.item(), 16)

    def test_positive_chain(self):
        a, b, c = Node(2), Node(3), Node(4)
        d = a + b * c
        e = d.relu()
        e.backward()
        self.assertEqual(e.data.item(), 14)
        self.assertEqual([a.grad.item(), b.grad.item(), c.grad.item(), d.grad.item(), e.grad.item()], [1, 4, 3, 1, 1])

    def test_assignment_example(self):
        a, b, c = Node(2), Node(-3), Node(10)
        d = a + b * c
        e = d.relu()
        e.backward()
        self.assertEqual(d.data.item(), -28)
        self.assertEqual(e.data.item(), 0)
        self.assertEqual([a.grad.item(), b.grad.item(), c.grad.item(), d.grad.item(), e.grad.item()], [0, 0, 0, 0, 1])

    def test_leaf(self):
        a = Node(5)
        a.backward()
        self.assertEqual(a.grad.item(), 1)

    def test_repeated_backward(self):
        a = Node(2)
        out = (a * a + a).relu()
        out.backward()
        out.backward()
        self.assertEqual(a.grad.item(), 5)
        self.assertEqual(out.grad.item(), 1)

    def test_finite_difference(self):
        def f(x):
            return (x[0] * x[1] + x[2]).clamp(min=0) + x[0] * x[0]

        for values in [[1.3, -0.7, 0.4], [1.3, -0.7, 2.0]]:
            with self.subTest(values=values):
                x = torch.tensor(values, dtype=torch.float64)
                a, b, c = [Node(value) for value in values]
                out = (a * b + c).relu() + a * a
                out.backward()
                for i, node in enumerate([a, b, c]):
                    step = torch.zeros_like(x)
                    step[i] = 1e-6
                    numerical = (f(x + step) - f(x - step)) / 2e-6
                    torch.testing.assert_close(node.grad, numerical, rtol=1e-7, atol=1e-7)

In [5]:
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestNode)
result = unittest.TextTestRunner(verbosity=2).run(suite)

test_add (__main__.TestNode.test_add) ... 

ok


test_assignment_example (__main__.TestNode.test_assignment_example) ... 

ok


test_constants (__main__.TestNode.test_constants) ... 

...

test_same_operand_mul (__main__.TestNode.test_same_operand_mul) ... 

ok


test_shared_graph (__main__.TestNode.test_shared_graph) ... 

ok


----------------------------------------------------------------------
Ran 12 tests in 0.429s

OK
